# Imports and Setup


In [1]:
import pandas as pd
from sodapy import Socrata

APP_KEY = "t2GPRE1GXlktxvqOIFMelPIvT"

client = Socrata(
    "data.cityofnewyork.us",
    APP_KEY,
    timeout=90
)

RELEASE_NAMES = ["prelim", "exec", "adopt"]

# Expense Budget


https://data.cityofnewyork.us/api/v3/views/mwzb-yiwb/query.json


In [3]:
EXPENSE_DATASET_ID = "mwzb-yiwb"

### Check Fiscal Year


In [4]:
# 1. Fetch the 2 most recent unique fiscal years present in the dataset
year_query = "SELECT fiscal_year GROUP BY fiscal_year ORDER BY fiscal_year DESC LIMIT 2"
current_fy_json = client.get(EXPENSE_DATASET_ID, query=year_query)

In [5]:
# Extract the years into a list, e.g., ['2026', '2025']
planned_fy = [row['fiscal_year'] for row in current_fy_json][0]

planned_fy_name = f"FY{int(planned_fy) % 100}"

print(planned_fy)
print(planned_fy_name)

print()

current_fy = int(planned_fy) - 1
current_fy_name = f"FY{int(current_fy) % 100}"

print(current_fy)
print(current_fy_name)

2027
FY27

2026
FY26


In [ ]:
# str_sum_current_adopted_amount = f"{current_fy_name}_current_adopted_amount"
# str_sum_current_modified_amount = f"{current_fy_name}_current_modified_amount"
# str_sum_financial_plan_amount = f"{planned_fy_name}_financial_plan_amount"

# str_sum_current_adopted_position = f"{current_fy_name}_current_adopted_position"
# str_sum_current_modified_position = f"{current_fy_name}_current_modified_position"
# str_sum_financial_plan_position = f"{planned_fy_name}_financial_plan_position"


str_sum_current_adopted_amount = "adopted_budget_amount"
str_sum_current_modified_amount = "current_modified_budget_amount"
str_sum_financial_plan_amount = "financial_plan_amount"

str_sum_current_adopted_position = "adopted_budget_position"
str_sum_current_modified_position = "current_modified_budget_position"
str_sum_financial_plan_position = "financial_plan_position"

select_arr = [
    "publication_date",
    "fiscal_year",
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "budget_code_number",
    "budget_code_name",
    "financial_plan_savings_flag",
    
    # f"adopted_budget_amount AS {str_sum_current_adopted_amount}",
    # f"current_modified_budget_amount AS {str_sum_current_modified_amount}",
    # f"financial_plan_amount AS {str_sum_financial_plan_amount}",
    # f"adopted_budget_position AS {str_sum_current_adopted_position}",
    # f"current_modified_budget_position AS {str_sum_current_modified_position}",
    # f"financial_plan_position AS {str_sum_financial_plan_position}"
    
    # f"SUM(adopted_budget_amount) AS {str_sum_current_adopted_amount}",
    # f"SUM(current_modified_budget_amount) AS {str_sum_current_modified_amount}",
    # f"SUM(financial_plan_amount) AS {str_sum_financial_plan_amount}",
    # f"SUM(adopted_budget_position) AS {str_sum_current_adopted_position}",
    # f"SUM(current_modified_budget_position) AS {str_sum_current_modified_position}",
    # f"SUM(financial_plan_position) AS {str_sum_financial_plan_position}"
]

select_string = ", ".join(select_arr)

print(f"select_string:\n{select_string}\n")

where_string = f"fiscal_year IN ({planned_fy})"

print(f"where_string:\n{where_string}\n")


groupby_arr = [
    "publication_date",
    "fiscal_year",
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "budget_code_number",
    "budget_code_name",
    "financial_plan_savings_flag"
]

groupby_string = ", ".join(groupby_arr)


print(f"groupby_string:\n{groupby_string}\n")



orderby_arr = [
    "agency_number",
    "unit_appropriation_number",
    "budget_code_number",
    "publication_date DESC"
]

orderby_string = ", ".join(orderby_arr)

print(f"orderby_string:\n{orderby_string}\n")



limit = 1000

select_string:
publication_date, fiscal_year, agency_number, agency_name, unit_appropriation_number, unit_appropriation_name, budget_code_number, budget_code_name, financial_plan_savings_flag

where_string:
fiscal_year IN (2027)

groupby_string:
publication_date, fiscal_year, agency_number, agency_name, unit_appropriation_number, unit_appropriation_name, budget_code_number, budget_code_name, financial_plan_savings_flag

orderby_string:
agency_number, unit_appropriation_number, budget_code_number, publication_date DESC



### Query Data if necessary


In [ ]:
# query_string = f"SELECT {select_string} WHERE {where_string} GROUP BY {groupby_string} ORDER BY {orderby_string} LIMIT {limit}"
# print(query_string)

data = []
offset = 0


# max_item = 0
try:
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}.csv")
except:
    print(f"Querying for fiscal year {planned_fy}:\n")
    while True:
        print(f"Fetching rows starting at offset: {offset}")

        temp = client.get(
            dataset_identifier=EXPENSE_DATASET_ID,
            # select=select_string,
            where=where_string,
            # group=groupby_string,
            # order=orderby_string,
            limit=limit,
            offset=offset
        )
        
        if not temp:
            break
        
        # print(temp[0])
        
        data.extend(temp)
        
        offset += limit
            
    print(f"End of query. Successfully fetched {len(data)} total rows.")
    
    len(data)
    
    # Convert to pandas DataFrame
    df = pd.DataFrame.from_records(data)
    # df.drop(columns=["financial_plan_savings_flag"], inplace=True)
    
    df.to_csv(f"./.data_cache/backup_{planned_fy}.csv", index=False)    

In [ ]:
# for i, col in enumerate(df.columns):
#     print(f"{i}: {col}")

0. "publication_date"
1. "fiscal_year"
2. "agency_number"
3. "agency_name"
4. "unit_appropriation_number"
5. "unit_appropriation_name"
6. "responsibility_center_code"
7. "responsibility_center_name"
8. "budget_code_number"
9. "budget_code_name"
10. "object_class_number"
11. "object_class_name"
12. "object_code"
13. "object_code_name"
14. "financial_plan_savings_flag"
15. "intra_city_purchase_code"
16. "adopted_budget_amount"
17. "current_modified_budget_amount"
18. "financial_plan_amount"
19. "adopted_budget_position"
20. "current_modified_budget_position"
21. "financial_plan_position"
22. "adopted_budget_number_of_contracts"
23. "current_modified_budget_number_of_contracts"
24. "financial_plan_number_of_contracts"


### Convert to ints


In [ ]:
numeric_cols = [
    "adopted_budget_amount",
    "current_modified_budget_amount",
    "financial_plan_amount",
    
    "adopted_budget_position",
    "current_modified_budget_position",
    "financial_plan_position",
    
    "adopted_budget_number_of_contracts",
    "current_modified_budget_number_of_contracts",
    "financial_plan_number_of_contracts"
]


for i, col in enumerate(numeric_cols):
    print(f"{i}: {col}")
    print(df[col].dtype)
    
    df[col] = df[col].astype(int)
    print(df[col].dtype)
    print()

In [ ]:
df = df.sort_values(
    [
        "agency_number",
        "unit_appropriation_number",
        "responsibility_center_code",
        "budget_code_number",
        "object_class_number",
        "object_code",
        "publication_date"
        ],
    ascending=[True, True, True, True, True, True, False]
    ).reset_index(drop=True)

df.head()

## Track releases


In [ ]:
pub_dates = df["publication_date"].sort_values(ascending=True).unique().tolist()

num_pubs_this_year = len(pub_dates)

print(f"{num_pubs_this_year} pub_dates in FY {planned_fy}: {pub_dates}")

## Set up base DataFrame and map function


In [ ]:
base_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    "financial_plan_savings_flag",
    "intra_city_purchase_code",
]

base_df = df[base_cols].drop_duplicates().reset_index(drop=True)
base_df.head()

In [ ]:
# AGENCY_MAP = df[["agency_number","agency_name"]].drop_duplicates().set_index('agency_number')['agency_name'].to_dict()

# UOA_MAP = df.groupby('agency_number')[['unit_appropriation_number', 'unit_appropriation_name']].apply(
#     lambda x: dict(zip(x['unit_appropriation_number'], x['unit_appropriation_name']))
# ).to_dict()

# BUDGETCODE_MAP = df.groupby(['agency_number', 'unit_appropriation_number'])[["budget_code_number","budget_code_name"]].apply(lambda x: dict(zip(x['budget_code_number'], x['budget_code_name']))).to_dict()

# BUDGETCODE_MAP

## Loop through each release


In [ ]:
print(pub_dates)

print(planned_fy)

for i, pub_date in enumerate(pub_dates):
    release_name = RELEASE_NAMES[i]
    
    ith_release_df = df[df['publication_date'] == pub_date]
    
    print(i, release_name)

## Collapse Releases


In [ ]:
df.head(10)

In [ ]:
# df.groupby(
#     by = [
#         "agency_number",
#         "agency_name",
#         "unit_appropriation_number",
#         "unit_appropriation_name",
#         "responsibility_center_code",
#         "responsibility_center_name",
#         "budget_code_number",
#         "budget_code_name",
#     ]
# )

In [ ]:
target_cols = [
    *base_cols,
    f"{str_sum_current_adopted_amount}",
    f"{str_sum_current_modified_amount}",
    f"{str_sum_financial_plan_amount}",
    
    f"{str_sum_current_adopted_position}",
    f"{str_sum_current_modified_position}",
    f"{str_sum_financial_plan_position}",
]


print(target_cols)

In [ ]:
collapsed_df = base_df.copy()

# collapsed_df.head()

for i, pub_date in enumerate(pub_dates):
    ith_release_df = df[df["publication_date"] == pub_date][target_cols].copy()
    
    print(i)
    
    if i <= 2:
        ith_release_df = ith_release_df.rename(
            columns={
                f"{str_sum_financial_plan_amount}":f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[i]}",
                f"{str_sum_financial_plan_position}":f"{str_sum_financial_plan_position}_{RELEASE_NAMES[i]}"
                }
        )
    else:
        raise Exception(f"Bad indexing, i:{i} >= num_pubs_this_year or i:{i} < 0")
    
    if i < num_pubs_this_year - 1:
        ith_release_df = ith_release_df.drop(columns=[
            f"{str_sum_current_adopted_amount}",
            f"{str_sum_current_modified_amount}",
            f"{str_sum_current_adopted_position}",
            f"{str_sum_current_modified_position}"
            ])
    
    print(f"[{i}/{num_pubs_this_year}] pub_date -- {pub_date} ({RELEASE_NAMES[i]}):")
    print(ith_release_df.columns)
    print()
    
    collapsed_df = collapsed_df.merge(right=ith_release_df,on=base_cols, how='left')
    
    # break

# collapsed_df = collapsed_df.fillna(0)

In [ ]:
print(collapsed_df.columns)

collapsed_df.head()

In [ ]:
collapsed_diff_df = collapsed_df.copy()

# for i in range(0,num_pubs_this_year):
    
#     # "total_adopted_budget_amount",
#     # "total_current_modified_budget_amount",
#     # "total_adopted_budget_position",
#     # "total_current_modified_budget_position"

#     str_sum_current_adopted_amount
#     str_sum_current_modified_amount

#     str_sum_current_adopted_position
#     str_sum_current_modified_position


#     amount_change_prev_adopt_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
#     # amount_change_percent_prev_adopt_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"

#     collapsed_diff_df[amount_change_prev_adopt_name]= collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount]
#     # collapsed_diff_df[amount_change_percent_prev_adopt_name]= (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount])/collapsed_df[i_amount_name]


#     amount_change_prev_modified_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"
#     # amount_change_percent_prev_modified_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

#     collapsed_diff_df[amount_change_prev_modified_name] = collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount]
#     # collapsed_diff_df[amount_change_percent_prev_modified_name] = (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount])/collapsed_df[i_amount_name]


#     position_change_prev_adopt_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
#     position_change_prev_modified_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

#     collapsed_diff_df[position_change_prev_adopt_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_adopted_position]
#     collapsed_diff_df[position_change_prev_modified_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_modified_position]

#     # collapsed_diff_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-adopt"] = (collapsed_df[i_position_name] - collapsed_df["total_adopted_budget_position"])/collapsed_df[i_position_name]
#     # collapsed_diff_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-modified"] = (collapsed_df[i_position_name] - collapsed_df["total_current_modified_budget_position"])/collapsed_df[i_position_name]
    
    
#     for j in range(0,i):
#         print(f"i={i}, j={j}")
#         print(f"{RELEASE_NAMES[i]} - {RELEASE_NAMES[j]}")
        
#         i_amount_name = f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[i]}"
#         j_amount_name = f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[j]}"
        
#         i_position_name = f"{str_sum_financial_plan_position}_{RELEASE_NAMES[i]}"
#         j_position_name = f"{str_sum_financial_plan_position}_{RELEASE_NAMES[j]}"
        
#         print()
#         print(f"i_amount_name: {i_amount_name}")
#         print(f"j_amount_name: {j_amount_name}")
#         print()
#         print(f"i_position_name: {i_position_name}")
#         print(f"j_position_name: {j_position_name}")
#         print()
            
            
#         amount_change_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
#         # amount_change_percent_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
        
#         print(amount_change_name)
#         # print(amount_change_percent_name)
            
#         collapsed_diff_df[amount_change_name] = collapsed_df[i_amount_name] - collapsed_df[j_amount_name]
#         # collapsed_diff_df[amount_change_percent_name] = (collapsed_df[i_amount_name] - collapsed_df[j_amount_name])/collapsed_df[i_amount_name]
        
        
#         print()
        
#         position_change_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
#         # position_change_percent_name = f"{str_sum_financial_plan_position}_%change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
        
#         print(position_change_name)
#         # print(position_change_percent_name)
        
#         collapsed_diff_df[position_change_name] = collapsed_df[i_position_name] - collapsed_df[j_position_name]
#         # collapsed_diff_df[position_change_percent_name] = (collapsed_df[i_position_name] - collapsed_df[j_position_name])/collapsed_df[i_position_name]
        
#         print("="*50)

In [ ]:
collapsed_diff_df

In [ ]:
# # "total_adopted_budget_amount",
# # "total_current_modified_budget_amount",
# # "total_adopted_budget_position",
# # "total_current_modified_budget_position"

# str_sum_current_adopted_amount
# str_sum_current_modified_amount

# str_sum_current_adopted_position
# str_sum_current_modified_position


# amount_change_prev_adopt_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
# amount_change_percent_prev_adopt_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"

# collapsed_diff_df[amount_change_prev_adopt_name]= collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount]
# collapsed_diff_df[amount_change_percent_prev_adopt_name]= (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount])/collapsed_df[i_amount_name]


# amount_change_prev_modified_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"
# amount_change_percent_prev_modified_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

# collapsed_diff_df[amount_change_prev_modified_name] = collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount]
# collapsed_diff_df[amount_change_percent_prev_modified_name] = (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount])/collapsed_df[i_amount_name]


# position_change_prev_adopt_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
# position_change_prev_modified_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

# collapsed_diff_df[position_change_prev_adopt_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_adopted_position]
# collapsed_diff_df[position_change_prev_modified_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_modified_position]

# # collapsed_diff_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-adopt"] = (collapsed_df[i_position_name] - collapsed_df["total_adopted_budget_position"])/collapsed_df[i_position_name]
# # collapsed_diff_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-modified"] = (collapsed_df[i_position_name] - collapsed_df["total_current_modified_budget_position"])/collapsed_df[i_position_name]

In [ ]:
collapsed_diff_df

In [ ]:
for c in collapsed_diff_df.columns:
    print(c)

In [ ]:
collapsed_diff_df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="Raw",index=False)

## Units of Appropriation


In [ ]:
UoA_base_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "financial_plan_savings_flag",
]

UoA_collapsed_df = collapsed_diff_df.groupby(UoA_base_cols).sum().reset_index()

# target_cols = UoA_collapsed_df.columns[14:]
# target_cols = UoA_collapsed_df.columns[7:]

# print(target_cols)

# UoA_collapsed_df = UoA_collapsed_df[[*UoA_base_cols,*target_cols]]

UoA_collapsed_df

In [ ]:
UoA_collapsed_df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="UoA",index=False)

## Agencies


In [ ]:
Agency_base_cols = [
    "agency_number",
    "agency_name",
]

Agency_collapsed_df = collapsed_diff_df.groupby(Agency_base_cols).sum().reset_index()

# target_cols = Agency_collapsed_df.columns[14:]

# print(target_cols)

Agency_collapsed_df = Agency_collapsed_df[[*Agency_base_cols,*target_cols]]

Agency_collapsed_df

In [ ]:
Agency_collapsed_df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="Agency",index=False)